#Instalación y verificación del entorno

In [ ]:
!pip install torchmetrics -q

import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random
import os

# Verificar entorno
print("=== Entorno ===")
print(f"PyTorch:     {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsando: {DEVICE}")


#Descarga y carga del dataset Penn-Fudan


In [ ]:
import urllib.request
import zipfile

# Penn-Fudan no está en torchvision.datasets como descarga directa,
# lo descargamos manualmente desde el repositorio oficial
URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
ZIP_PATH = "PennFudanPed.zip"
DATA_DIR = "./data"

if not os.path.exists(os.path.join(DATA_DIR, "PennFudanPed")):
    print("Descargando Penn-Fudan...")
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATA_DIR)
    os.remove(ZIP_PATH)
    print("Descarga completada")
else:
    print("Dataset ya descargado")

# Verificar estructura
img_dir  = os.path.join(DATA_DIR, "PennFudanPed", "PNGImages")
mask_dir = os.path.join(DATA_DIR, "PennFudanPed", "PedMasks")
imgs  = sorted(os.listdir(img_dir))
masks = sorted(os.listdir(mask_dir))

print(f"\nImágenes encontradas: {len(imgs)}")
print(f"Máscaras encontradas: {len(masks)}")
print(f"\nEjemplos de imágenes: {imgs[:3]}")
print(f"Ejemplos de máscaras: {masks[:3]}")

#Clase Dataset + visualización Ground Truth

In [ ]:
class PennFudanDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir  = img_dir
        self.mask_dir = mask_dir
        self.imgs  = sorted(os.listdir(img_dir))
        self.masks = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        # Cargar imagen y máscara
        img  = Image.open(os.path.join(self.img_dir,  self.imgs[idx])).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, self.masks[idx]))
        mask = np.array(mask)  # valores únicos = instancias distintas

        # Extraer instancias (ignorar fondo = 0)
        obj_ids = np.unique(mask)
        obj_ids = obj_ids[obj_ids != 0]

        # Una máscara binaria por instancia
        masks_bin = (mask == obj_ids[:, None, None])  # shape: (N, H, W)

        # Cajas desde cada máscara
        boxes = []
        for m in masks_bin:
            pos = np.where(m)
            x1, y1 = int(pos[1].min()), int(pos[0].min())
            x2, y2 = int(pos[1].max()), int(pos[0].max())
            boxes.append([x1, y1, x2, y2])

        target = {
            "boxes":  torch.tensor(boxes,     dtype=torch.float32),
            "labels": torch.ones(len(boxes),  dtype=torch.int64),
            "masks":  torch.tensor(masks_bin, dtype=torch.uint8),
        }
        return img, target


# Instanciar dataset completo
dataset = PennFudanDataset(img_dir, mask_dir)
print(f"Total imágenes en dataset: {len(dataset)}")

# --- Split train / val (80/20) ---
random.seed(42)
n = len(dataset)
idx = list(range(n))
random.shuffle(idx)
cut = int(0.8 * n)
train_idx, valid_idx = idx[:cut], idx[cut:]
valid_set = torch.utils.data.Subset(dataset, valid_idx)
print(f"Train: {len(train_idx)} imágenes | Val: {len(valid_idx)} imágenes")


# VISUALIZACIÓN GROUND TRUTH — 3 imágenes


fig, axes = plt.subplots(3, 2, figsize=(12, 14))
fig.suptitle("Ground Truth — Penn-Fudan (Detección + Segmentación)", fontsize=14, fontweight='bold')

for row, sample_idx in enumerate([0, 1, 2]):
    img, target = dataset[valid_idx[sample_idx]]
    img_np = np.array(img)
    boxes  = target["boxes"]
    masks  = target["masks"]

    # ---- Columna izquierda: imagen + cajas GT ----
    ax = axes[row, 0]
    ax.imshow(img_np)
    for box in boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, "GT person", color='lime',
                fontsize=8, fontweight='bold',
                bbox=dict(facecolor='black', alpha=0.5, pad=1))
    ax.set_title(f"Imagen {sample_idx+1} — Cajas GT ({len(boxes)} personas)", fontsize=10)
    ax.axis('off')

    # ---- Columna derecha: imagen + máscaras GT superpuestas ----
    ax = axes[row, 1]
    ax.imshow(img_np)
    combined_mask = masks.numpy().sum(axis=0).clip(0, 1)  # fusión de instancias
    ax.imshow(combined_mask, alpha=0.45, cmap='Reds')
    ax.set_title(f"Imagen {sample_idx+1} — Máscaras GT ({len(masks)} instancias)", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig("gt_visualizacion.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: gt_visualizacion.png")

#Explicación teórica: clasificación vs detección vs segmentación

In [ ]:


explicacion = """
╔════════════════════════════════════════════════════════════╗
║         CLASIFICACIÓN vs DETECCIÓN vs SEGMENTACIÓN              ║
╠════════════════════════════════════════════════════════════╣
║                                                                  ║
║  CLASIFICACIÓN                                                   ║
║  → Entrada: imagen completa                                      ║
║  → Salida:  1 etiqueta global  (ej: "hay una persona")           ║
║  → No dice DÓNDE está el objeto, solo QUÉ hay en la imagen       ║
║                                                                  ║
║  DETECCIÓN                                                       ║
║  → Entrada: imagen completa                                      ║
║  → Salida:  N cajas (x1,y1,x2,y2) + etiqueta + score            ║
║  → Localiza DÓNDE está cada objeto con un rectángulo             ║
║  → En Penn-Fudan: una caja por peatón detectado                  ║
║                                                                  ║
║  SEGMENTACIÓN                                                    ║
║  → Entrada: imagen completa                                      ║
║  → Salida:  máscara H×W (un valor por píxel)                     ║
║  → Dice exactamente QUÉ píxeles pertenecen a cada objeto         ║
║  → En Penn-Fudan: máscara binaria por instancia de peatón        ║
║                                                                  ║
║  DIFERENCIA CLAVE                                                ║
║  Clasificación < Detección < Segmentación en nivel de detalle.   ║
║  Cada nivel añade información espacial: primero la clase,        ║
║  luego la región rectangular, finalmente el contorno exacto.     ║
║  A mayor detalle, mayor coste computacional y más anotaciones    ║
║  necesarias para entrenar.                                       ║
╚════════════════════════════════════════════════════════════╝
"""
print(explicacion)

# Figura resumen visual de los tres niveles
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Los tres niveles de análisis visual", fontsize=13, fontweight='bold')

img, target = dataset[valid_idx[0]]
img_np = np.array(img)
boxes  = target["boxes"]
masks  = target["masks"]

# --- Panel 1: Clasificación ---
ax = axes[0]
ax.imshow(img_np)
ax.set_title("Clasificación\nSalida: 'person'", fontsize=10)
ax.axis('off')

# --- Panel 2: Detección ---
ax = axes[1]
ax.imshow(img_np)
for box in boxes:
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-5, "person", color='lime', fontsize=8, fontweight='bold',
            bbox=dict(facecolor='black', alpha=0.5, pad=1))
ax.set_title("Detección\nSalida: cajas (x1,y1,x2,y2) + score", fontsize=10)
ax.axis('off')

# --- Panel 3: Segmentación ---
ax = axes[2]
ax.imshow(img_np)
combined = masks.numpy().sum(axis=0).clip(0, 1)
ax.imshow(combined, alpha=0.5, cmap='Reds')
ax.set_title("Segmentación\nSalida: máscara píxel a píxel", fontsize=10)
ax.axis('off')

plt.tight_layout()
plt.savefig("explicacion_niveles.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: explicacion_niveles.png")


Carga de Faster R-CNN e inferencia sobre validación


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision import transforms

# --- Cargar modelo preentrenado (COCO, 91 clases) ---
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model_det = fasterrcnn_resnet50_fpn(weights=weights)
model_det.to(DEVICE)
model_det.eval()
print(f"Faster R-CNN cargado en {DEVICE}")
print(f"   Backbone: ResNet-50 + FPN")
print(f"   Head:     RPN + RoI Heads (clasificación + regresión de cajas)")

# --- Parámetros ---
SCORE_THR = 0.5   # umbral de score para filtrar predicciones
PERSON_ID = 1     # en COCO, clase 1 = "person"

# --- Inferencia sobre las primeras 20 imágenes de validación ---

print("\nEjecutando inferencia sobre validación...")

all_preds   = []   # predicciones filtradas por imagen
all_targets = []   # GT por imagen

for i in range(20):
    img, target = dataset[valid_idx[i]]
    img_t = transforms.ToTensor()(img).to(DEVICE)

    with torch.no_grad():
        out = model_det([img_t])[0]

    # Filtrar: solo clase "person" con score >= umbral
    keep = (out['scores'] >= SCORE_THR) & (out['labels'] == PERSON_ID)

    preds_i = {
        'boxes':  out['boxes'][keep].cpu(),
        'scores': out['scores'][keep].cpu(),
        'labels': out['labels'][keep].cpu(),
    }
    all_preds.append(preds_i)
    all_targets.append({
        'boxes':  target['boxes'].cpu(),
        'labels': target['labels'].cpu(),
        'masks':  target['masks'].cpu(),
    })

    n_pred = keep.sum().item()
    n_gt   = len(target['boxes'])
    print(f"  Img {i:2d}: {n_pred} predicciones | {n_gt} GT")

print(f"\n Inferencia completada sobre {len(all_preds)} imágenes")

#Faster R-CNN — Visualización de predicciones

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
fig.suptitle("Faster R-CNN — GT vs Predicción", fontsize=14, fontweight='bold')

# Indices elegidos: variedad de casos (normal, muchas pred, pocas pred)
sample_indices = [0, 5, 16]

for row, i in enumerate(sample_indices):
    img, target = dataset[valid_idx[i]]
    img_np = np.array(img)
    boxes_gt   = target['boxes'].numpy()
    boxes_pred = all_preds[i]['boxes'].numpy()
    scores     = all_preds[i]['scores'].numpy()

    # columna izquierda: ground truth
    ax = axes[row, 0]
    ax.imshow(img_np)
    for box in boxes_gt:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-5, "GT person", color='lime', fontsize=8,
                fontweight='bold',
                bbox=dict(facecolor='black', alpha=0.5, pad=1))
    ax.set_title(f"Imagen {i} — GT ({len(boxes_gt)} personas)", fontsize=10)
    ax.axis('off')

    # columna derecha: predicciones del modelo
    ax = axes[row, 1]
    ax.imshow(img_np)
    for box, score in zip(boxes_pred, scores):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"pred {score:.2f}", color='red', fontsize=8,
                fontweight='bold',
                bbox=dict(facecolor='black', alpha=0.5, pad=1))
    ax.set_title(
        f"Imagen {i} — Pred ({len(boxes_pred)} detecciones, thr={SCORE_THR})",
        fontsize=10
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig("predicciones_deteccion.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: predicciones_deteccion.png")

#Tabla de predicciones — score, cajas, IoU y TP/FP

In [ ]:
import pandas as pd

def iou_box(a, b):
    x_left   = max(a[0], b[0]);  y_top    = max(a[1], b[1])
    x_right  = min(a[2], b[2]);  y_bottom = min(a[3], b[3])
    inter    = max(0, x_right - x_left) * max(0, y_bottom - y_top)
    area_a   = (a[2]-a[0]) * (a[3]-a[1])
    area_b   = (b[2]-b[0]) * (b[3]-b[1])
    union    = area_a + area_b - inter
    return 0.0 if union == 0 else inter / union

def best_iou_with_gt(pred_box, gt_boxes):
    # devuelve la caja GT con mayor IoU y su valor
    if len(gt_boxes) == 0:
        return None, 0.0
    ious = [iou_box(pred_box, gt) for gt in gt_boxes]
    best_idx = int(np.argmax(ious))
    return gt_boxes[best_idx], ious[best_idx]

rows = []
for i in range(20):
    boxes_pred = all_preds[i]['boxes'].numpy()
    scores     = all_preds[i]['scores'].numpy()
    boxes_gt   = all_targets[i]['boxes'].numpy()

    for pred_box, score in zip(boxes_pred, scores):
        gt_box, iou = best_iou_with_gt(pred_box, boxes_gt)

        # formatear cajas como strings legibles
        pb = f"({pred_box[0]:.0f},{pred_box[1]:.0f},{pred_box[2]:.0f},{pred_box[3]:.0f})"
        if gt_box is not None:
            gb = f"({gt_box[0]:.0f},{gt_box[1]:.0f},{gt_box[2]:.0f},{gt_box[3]:.0f})"
        else:
            gb = "—"

        rows.append({
            "img":        i,
            "score":      round(float(score), 3),
            "box_pred":   pb,
            "box_gt":     gb,
            "IoU":        round(iou, 3),
            "TP/FP":      "TP" if iou >= 0.5 else "FP",
        })

df = pd.DataFrame(rows)

# mostrar primeras 15 filas en pantalla
print(df[["img","score","box_pred","box_gt","IoU","TP/FP"]].head(15).to_string(index=False))
print(f"\nTotal predicciones: {len(df)}")
print(f"TP: {(df['TP/FP']=='TP').sum()}  |  FP: {(df['TP/FP']=='FP').sum()}")

# guardar CSV para adjuntar a la entrega
df.to_csv("tabla_predicciones.csv", index=False)
print("Tabla guardada: tabla_predicciones.csv")

#IoU calculado a mano — 2 casos paso a paso

In [ ]:
print("=" * 60)
print("CASO 1 — TP esperado (IoU alto)")
print("=" * 60)

a1 = [95, 82, 210, 352]    # caja predicha  (img 0)
b1 = [97, 78, 209, 358]    # caja GT        (img 0)

x_left1   = max(a1[0], b1[0]);  print(f"x_left   = max({a1[0]}, {b1[0]}) = {x_left1}")
y_top1    = max(a1[1], b1[1]);  print(f"y_top    = max({a1[1]}, {b1[1]}) = {y_top1}")
x_right1  = min(a1[2], b1[2]);  print(f"x_right  = min({a1[2]}, {b1[2]}) = {x_right1}")
y_bottom1 = min(a1[3], b1[3]);  print(f"y_bottom = min({a1[3]}, {b1[3]}) = {y_bottom1}")

inter_w1 = max(0, x_right1 - x_left1)
inter_h1 = max(0, y_bottom1 - y_top1)
inter1   = inter_w1 * inter_h1
print(f"\nInterseccion: {inter_w1} x {inter_h1} = {inter1} px²")

area_a1 = (a1[2]-a1[0]) * (a1[3]-a1[1])
area_b1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
union1  = area_a1 + area_b1 - inter1
print(f"Area pred:    ({a1[2]}-{a1[0]}) x ({a1[3]}-{a1[1]}) = {area_a1} px²")
print(f"Area GT:      ({b1[2]}-{b1[0]}) x ({b1[3]}-{b1[1]}) = {area_b1} px²")
print(f"Union:        {area_a1} + {area_b1} - {inter1} = {union1} px²")

iou1 = inter1 / union1
print(f"\nIoU = {inter1} / {union1} = {iou1:.3f}")
print(f"Criterio IoU >= 0.5 --> {'TP' if iou1 >= 0.5 else 'FP'}")

print()
print("=" * 60)
print("CASO 2 — FP esperado (IoU bajo)")
print("=" * 60)

a2 = [191, 89, 238, 263]   # caja predicha  (img 1)
b2 = [147, 84, 238, 365]   # caja GT        (img 1)

x_left2   = max(a2[0], b2[0]);  print(f"x_left   = max({a2[0]}, {b2[0]}) = {x_left2}")
y_top2    = max(a2[1], b2[1]);  print(f"y_top    = max({a2[1]}, {b2[1]}) = {y_top2}")
x_right2  = min(a2[2], b2[2]);  print(f"x_right  = min({a2[2]}, {b2[2]}) = {x_right2}")
y_bottom2 = min(a2[3], b2[3]);  print(f"y_bottom = min({a2[3]}, {b2[3]}) = {y_bottom2}")

inter_w2 = max(0, x_right2 - x_left2)
inter_h2 = max(0, y_bottom2 - y_top2)
inter2   = inter_w2 * inter_h2
print(f"\nInterseccion: {inter_w2} x {inter_h2} = {inter2} px²")

area_a2 = (a2[2]-a2[0]) * (a2[3]-a2[1])
area_b2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
union2  = area_a2 + area_b2 - inter2
print(f"Area pred:    ({a2[2]}-{a2[0]}) x ({a2[3]}-{a2[1]}) = {area_a2} px²")
print(f"Area GT:      ({b2[2]}-{b2[0]}) x ({b2[3]}-{b2[1]}) = {area_b2} px²")
print(f"Union:        {area_a2} + {area_b2} - {inter2} = {union2} px²")

iou2 = inter2 / union2
print(f"\nIoU = {inter2} / {union2} = {iou2:.3f}")
print(f"Criterio IoU >= 0.5 --> {'TP' if iou2 >= 0.5 else 'FP'}")

print()
print("Verificacion con funcion iou_box:")
print(f"  Caso 1: {iou_box(a1, b1):.3f}")
print(f"  Caso 2: {iou_box(a2, b2):.3f}")

#mAP@0.5 con torchmetrics

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(iou_type='bbox')

# alimentar la metrica con todas las imagenes de validacion
for i in range(20):
    boxes_pred = all_preds[i]['boxes']
    scores     = all_preds[i]['scores']
    labels_pred = all_preds[i]['labels']
    boxes_gt   = all_targets[i]['boxes']
    labels_gt  = all_targets[i]['labels']

    # formato que espera torchmetrics
    pred_entry = [{
        'boxes':  boxes_pred,
        'scores': scores,
        'labels': labels_pred,
    }]
    target_entry = [{
        'boxes':  boxes_gt,
        'labels': labels_gt,
    }]

    metric.update(pred_entry, target_entry)

results = metric.compute()

print("Resultados mAP:")
print(f"  mAP@0.5      = {results['map_50'].item():.4f}")
print(f"  mAP@0.5:0.95 = {results['map'].item():.4f}")
print(f"  mAR@100      = {results['mar_100'].item():.4f}")
print()
print("Interpretacion:")
print(f"  Un mAP@0.5 de {results['map_50'].item():.4f} significa que el modelo")
print(f"  acierta aproximadamente el {results['map_50'].item()*100:.1f}% de las")
print(f"  detecciones considerando un umbral de IoU >= 0.5")

# Interpretación de resultados — mAP, umbral y tipos de error

## mAP@0.5 = 0.9917

Este valor cercano a 1.0 indica que el modelo, preentrenado en COCO, generaliza muy bien a Penn-Fudan porque la clase *person* está representada de forma similar en ambos: peatones en posición vertical, bien iluminados y sin oclusiones severas.

## Efecto del umbral de score

- **Umbral actual 0.5** → 88 predicciones (51 TP, 37 FP)
- **Si subimos el umbral** (ej. 0.9): reducimos falsos positivos pero aumentamos falsos negativos, perdiendo detecciones de personas parcialmente ocluidas o en segundo plano.
- **Si bajamos el umbral** (ej. 0.3): recuperamos esas detecciones (menos FN) pero aparecen más FP, especialmente en zonas de fondo con textura similar a la humana.

## Errores observados

**Falsos positivos (FP = 37):** el caso más llamativo es la imagen 16, donde el modelo genera 19 predicciones para solo 3 GT. El modelo detecta partes del cuerpo (cabeza, torso) como instancias independientes o confunde estructuras del fondo con personas.

**Falsos negativos:** el mAR@100 = 0.8627 indica que aproximadamente un 14% de las personas GT no son detectadas, probablemente por oclusión parcial o tamaño reducido en la imagen.

## Conclusión parcial

Un mAP alto no implica ausencia de errores. Los 37 FP revelan que el umbral de score por defecto (0.5) es demasiado permisivo para algunas imágenes con múltiples personas superpuestas o fondo complejo.





#DeepLabV3 — Carga del modelo e inferencia de segmentación


In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from torchvision import transforms

# clase "person" en Pascal VOC (esquema de clases que usa DeepLabV3 preentrenado)
PERSON_CLASS = 15

# cargar modelo preentrenado
weights_seg = DeepLabV3_ResNet50_Weights.DEFAULT
model_seg = deeplabv3_resnet50(weights=weights_seg)
model_seg.to(DEVICE)
model_seg.eval()
print(f"DeepLabV3 cargado en {DEVICE}")
print(f"Backbone: ResNet-50 | Head: ASPP + clasificador por pixel")
print(f"Clases: 21 (Pascal VOC) | Clase person = {PERSON_CLASS}")

# inferencia sobre las mismas 20 imagenes de validacion
print("\nEjecutando inferencia de segmentacion...")

all_pred_masks = []   # mascaras binarias predichas
all_gt_masks   = []   # mascaras binarias GT (union de instancias)

to_tensor = transforms.ToTensor()

for i in range(20):
    img, target = dataset[valid_idx[i]]
    img_t = to_tensor(img).to(DEVICE)

    with torch.no_grad():
        out = model_seg(img_t.unsqueeze(0))['out'][0]   # [21, H, W]

    # mascara predicha: pixeles clasificados como person
    pred_class = out.argmax(0).cpu().numpy()            # [H, W]
    pred_bin   = (pred_class == PERSON_CLASS).astype(np.uint8)

    # mascara GT: union de todas las instancias de la imagen
    gt_bin = target['masks'].numpy().sum(axis=0).clip(0, 1).astype(np.uint8)

    all_pred_masks.append(pred_bin)
    all_gt_masks.append(gt_bin)

    coverage_pred = pred_bin.sum()
    coverage_gt   = gt_bin.sum()
    print(f"  Img {i:2d}: pred={coverage_pred:6d} px | GT={coverage_gt:6d} px")

print(f"\nInferencia completada sobre {len(all_pred_masks)} imagenes")

# visualizacion: 3 imagenes con GT y prediccion superpuestas
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
fig.suptitle("DeepLabV3 — Mascara GT vs Mascara Predicha", fontsize=14, fontweight='bold')

for row, i in enumerate([0, 5, 16]):
    img_np  = np.array(dataset[valid_idx[i]][0])
    gt_bin  = all_gt_masks[i]
    pr_bin  = all_pred_masks[i]

    # columna izquierda: mascara GT
    ax = axes[row, 0]
    ax.imshow(img_np)
    ax.imshow(gt_bin, alpha=0.5, cmap='Greens')
    ax.set_title(f"Imagen {i} — Mascara GT", fontsize=10)
    ax.axis('off')

    # columna derecha: mascara predicha
    ax = axes[row, 1]
    ax.imshow(img_np)
    ax.imshow(pr_bin, alpha=0.5, cmap='Reds')
    ax.set_title(f"Imagen {i} — Mascara Predicha (DeepLabV3)", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig("predicciones_segmentacion.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: predicciones_segmentacion.png")

#Dice por imagen y media — segmentación binaria

In [ ]:
def dice_bin(gt, pr, eps=1e-8):
    gt    = gt.astype(np.uint8)
    pr    = pr.astype(np.uint8)
    inter = (gt & pr).sum()
    return (2 * inter + eps) / (gt.sum() + pr.sum() + eps)

# calcular Dice para cada imagen
dice_scores = []
print(f"{'Img':>4}  {'Dice':>6}  {'GT px':>8}  {'Pred px':>8}  {'Valoracion':>12}")
print("-" * 50)

for i in range(20):
    gt_bin = all_gt_masks[i]
    pr_bin = all_pred_masks[i]
    d      = dice_bin(gt_bin, pr_bin)
    dice_scores.append(d)

    if d >= 0.85:
        valoracion = "bueno"
    elif d >= 0.65:
        valoracion = "aceptable"
    else:
        valoracion = "deficiente"

    print(f"{i:>4}  {d:>6.4f}  {gt_bin.sum():>8}  {pr_bin.sum():>8}  {valoracion:>12}")

dice_mean = np.mean(dice_scores)
dice_std  = np.std(dice_scores)
dice_min  = np.min(dice_scores)
dice_max  = np.max(dice_scores)

print("-" * 50)
print(f"\nDice medio:  {dice_mean:.4f}")
print(f"Desv. std:   {dice_std:.4f}")
print(f"Minimo:      {dice_min:.4f}  (img {int(np.argmin(dice_scores))})")
print(f"Maximo:      {dice_max:.4f}  (img {int(np.argmax(dice_scores))})")

# figura de barras para visualizar la distribucion de Dice
fig, ax = plt.subplots(figsize=(12, 4))
colores = ['green' if d >= 0.85 else 'orange' if d >= 0.65 else 'red'
           for d in dice_scores]
ax.bar(range(20), dice_scores, color=colores, edgecolor='black', linewidth=0.5)
ax.axhline(dice_mean, color='blue', linestyle='--', linewidth=1.5,
           label=f'Media = {dice_mean:.4f}')
ax.axhline(0.85, color='green', linestyle=':', linewidth=1,
           label='Umbral bueno (0.85)')
ax.axhline(0.65, color='orange', linestyle=':', linewidth=1,
           label='Umbral aceptable (0.65)')
ax.set_xlabel("Indice de imagen (validacion)")
ax.set_ylabel("Dice")
ax.set_title("Dice por imagen — DeepLabV3 sobre Penn-Fudan")
ax.set_xticks(range(20))
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig("dice_por_imagen.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: dice_por_imagen.png")

# Interpretación del coeficiente Dice — segmentación

## Resultado obtenido

Dice medio = 0.8929 sobre 20 imágenes de validación (σ = 0.0663, mín = 0.6800, máx = 0.9484).

## ¿Qué significa este valor?

El coeficiente Dice mide el solapamiento entre la máscara predicha y el ground truth, tomando valores entre 0 (sin solapamiento) y 1 (coincidencia perfecta). Un Dice de 0.89 se considera alto: significa que aproximadamente el 89% de los píxeles de persona son correctamente identificados por el modelo.

## ¿Cuándo es alto y cuándo es bajo?

- Dice ≥ 0.85 → segmentación buena. El modelo captura bien el contorno de las personas (17 de 20 imágenes en este experimento).
- Dice entre 0.65 y 0.85 → segmentación aceptable. Hay errores en bordes o zonas ocluidas, pero la región principal se detecta (imágenes 6, 7 y 16).
- Dice < 0.65 → segmentación deficiente. El modelo falla en localizar la región o genera muchos píxeles falsos.

## Casos problemáticos

La imagen 6 (Dice = 0.6800) y la imagen 7 (Dice = 0.7377) presentan el mayor error. En ambas, la predicción sobreestima significativamente los píxeles de persona (pred > GT), lo que indica que DeepLabV3 clasifica como persona píxeles del fondo cercano, posiblemente por similitud de textura o color con la ropa de los peatones.



#Análisis de errores — evidencia visual (3 fallos)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 14))
fig.suptitle("Analisis de errores — 3 fallos identificados", fontsize=14, fontweight='bold')

# --- FALLO 1: imagen 16 — exceso de predicciones (deteccion) ---
i = 16
img_np     = np.array(dataset[valid_idx[i]][0])
boxes_gt   = all_targets[i]['boxes'].numpy()
boxes_pred = all_preds[i]['boxes'].numpy()
scores     = all_preds[i]['scores'].numpy()

ax = axes[0, 0]
ax.imshow(img_np)
ax.set_title(f"Fallo 1 — Imagen {i}: GT ({len(boxes_gt)} personas)", fontsize=9)
for box in boxes_gt:
    x1,y1,x2,y2 = box
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                 linewidth=2, edgecolor='lime', facecolor='none'))
ax.axis('off')

ax = axes[0, 1]
ax.imshow(img_np)
ax.set_title(f"Fallo 1 — Imagen {i}: Pred ({len(boxes_pred)} detecciones)", fontsize=9)
for box, sc in zip(boxes_pred, scores):
    x1,y1,x2,y2 = box
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,
                 linewidth=1.5, edgecolor='red', facecolor='none'))
    ax.text(x1, y1-4, f"{sc:.2f}", color='red', fontsize=7,
            bbox=dict(facecolor='black', alpha=0.4, pad=1))
ax.axis('off')

ax = axes[0, 2]
ax.axis('off')
ax.text(0.05, 0.95,
    "FALLO 1: Explosion de falsos positivos\n\n"
    "Que paso:\n19 predicciones para 3 personas reales.\nEl modelo duplica detecciones sobre las mismas personas.\n\n"
    "Hipotesis:\nEl modelo COCO no esta ajustado a grupos\nde personas juntas. Detecta partes del\ncuerpo como instancias separadas.\n\n"
    "Mejora propuesta:\nSubir umbral de score a 0.7-0.8 o aplicar\nNMS mas agresivo. Fine-tuning con imagenes\nde multiples personas superpuestas.",
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(facecolor='lightyellow', alpha=0.8, pad=6))

# --- FALLO 2: imagen 7 — sobreestimacion de mascara (segmentacion) ---
i = 7
img_np = np.array(dataset[valid_idx[i]][0])
gt_bin = all_gt_masks[i]
pr_bin = all_pred_masks[i]

ax = axes[1, 0]
ax.imshow(img_np)
ax.imshow(gt_bin, alpha=0.5, cmap='Greens')
ax.set_title(f"Fallo 2 — Imagen {i}: GT ({gt_bin.sum()} px)", fontsize=9)
ax.axis('off')

ax = axes[1, 1]
ax.imshow(img_np)
ax.imshow(pr_bin, alpha=0.5, cmap='Reds')
ax.set_title(f"Fallo 2 — Imagen {i}: Pred ({pr_bin.sum()} px) | Dice={dice_scores[i]:.4f}", fontsize=9)
ax.axis('off')

ax = axes[1, 2]
ax.axis('off')
ax.text(0.05, 0.95,
    "FALLO 2: Sobreestimacion de mascara\n\n"
    "Que paso:\nDice=0.7377. La prediccion cubre 23222 px\nfrente a 13692 px del GT (70% mas).\n\n"
    "Hipotesis:\nLa persona lleva ropa oscura similar al\nfondo. DeepLabV3 extiende la mascara\nhacia el entorno inmediato.\n\n"
    "Mejora propuesta:\nAugmentations de color (jitter) durante\nfine-tuning para que el modelo no dependa\ndel color de la ropa.",
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(facecolor='lightyellow', alpha=0.8, pad=6))

# --- FALLO 3: imagen 6 — peor Dice del conjunto ---
i = 6
img_np = np.array(dataset[valid_idx[i]][0])
gt_bin = all_gt_masks[i]
pr_bin = all_pred_masks[i]

ax = axes[2, 0]
ax.imshow(img_np)
ax.imshow(gt_bin, alpha=0.5, cmap='Greens')
ax.set_title(f"Fallo 3 — Imagen {i}: GT ({gt_bin.sum()} px)", fontsize=9)
ax.axis('off')

ax = axes[2, 1]
ax.imshow(img_np)
ax.imshow(pr_bin, alpha=0.5, cmap='Reds')
ax.set_title(f"Fallo 3 — Imagen {i}: Pred ({pr_bin.sum()} px) | Dice={dice_scores[i]:.4f}", fontsize=9)
ax.axis('off')

ax = axes[2, 2]
ax.axis('off')
ax.text(0.05, 0.95,
    "FALLO 3: Dice minimo del conjunto\n\n"
    "Que paso:\nDice=0.6800. La prediccion duplica casi\nlos pixeles GT (56908 vs 32665 px).\n\n"
    "Hipotesis:\nVarias personas en grupo compacto. El\nmodelo fusiona siluetas individuales en\nuna region unica sobredimensionada.\n\n"
    "Mejora propuesta:\nUsar segmentacion de instancias (Mask R-CNN)\nen lugar de segmentacion semantica para\ndiferenciar personas individuales.",
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(facecolor='lightyellow', alpha=0.8, pad=6))

plt.tight_layout()
plt.savefig("analisis_errores.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: analisis_errores.png")

# Conclusión final

En esta práctica se han aplicado dos modelos preentrenados sobre el dataset Penn-Fudan para abordar detección y segmentación de personas. Faster R-CNN alcanzó un mAP@0.5 de 0.9917, lo que refleja una excelente capacidad para localizar personas mediante cajas cuando el score supera el umbral de 0.5. Sin embargo, este umbral permisivo generó 37 falsos positivos sobre 88 predicciones totales, evidenciando que un mAP alto no garantiza ausencia de errores en imágenes con grupos compactos de personas.

El cálculo de IoU a mano en dos casos ilustró con claridad la diferencia entre un verdadero positivo (IoU = 0.940) y un falso positivo (IoU = 0.320): la caja predicha en el segundo caso apenas solapaba con el ground truth, por lo que no superó el criterio IoU ≥ 0.5. Subir el umbral de score reduciría los falsos positivos pero incrementaría los falsos negativos, perdiendo detecciones de personas parcialmente ocluidas.

En segmentación, DeepLabV3 obtuvo un Dice medio de 0.8929, con casos problemáticos en imágenes donde varias personas aparecen juntas. En esos casos el modelo, al realizar segmentación semántica, fusiona instancias individuales en una región continua sobredimensionada, alejándose del ground truth píxel a píxel. Para mejorar ambas tareas, las líneas de trabajo más prometedoras son el ajuste del umbral de score, la incorporación de augmentations de color y el fine-tuning con ejemplos de los patrones donde el modelo falla.

